In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from time import sleep

# === CONFIG ===
input_csv = r"C:\Android Mobile App\Step1_URL_Search\1 - Kotline, Java, Dart, Android - Stepwise_ API Check only\Step-2-URL_list.csv"
output_csv = r"C:\Android Mobile App\Step1_URL_Search\1 - Kotline, Java, Dart, Android - Stepwise_ API Check only\Step-2-URL_list_with_metadata.csv"

# === LOAD TOKENS ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 6)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("❌ No GitHub tokens found in All_Tokens.env")

token_index = 0
def get_headers():
    return {
        "Authorization": f"token {tokens[token_index]}",
        "Accept": "application/vnd.github.mercy-preview+json",
        "User-Agent": "repo-analyzer"
    }

def rotate_token():
    global token_index
    token_index = (token_index + 1) % len(tokens)

# === LOAD CSV ===
df = pd.read_csv(input_csv)
df['full_name'] = df['html_url'].str.extract(r"github\.com/([^/]+/[^/]+)")

# === METADATA COLLECTION ===
results = []

def fetch_repo_data(full_name):
    base_url = f"https://api.github.com/repos/{full_name}"
    topics_url = f"{base_url}/topics"
    readme_url = f"{base_url}/readme"

    try:
        r = requests.get(base_url, headers=get_headers())
        if r.status_code == 403:  # Rate limit
            rotate_token()
            sleep(1)
            r = requests.get(base_url, headers=get_headers())
        if r.status_code != 200:
            return [None]*7

        data = r.json()
        lang = data.get("language", "")
        fork = data.get("fork", False)
        archived = data.get("archived", False)
        stars = data.get("stargazers_count", 0)
        name = (data.get("name") or "").lower()
        desc = (data.get("description") or "").lower()

        # Check topics
        rt = requests.get(topics_url, headers=get_headers())
        topics = rt.json().get("names", []) if rt.status_code == 200 else []
        android_in_topic = "android" in topics

        # Check README for "android"
        rr = requests.get(readme_url, headers=get_headers())
        android_in_readme = False
        if rr.status_code == 200:
            readme = rr.json().get("content", "")
            if readme:
                import base64
                decoded = base64.b64decode(readme).decode('utf-8', errors='ignore').lower()
                android_in_readme = "android" in decoded

        android_in_name_desc = "android" in name or "android" in desc
        return [lang, fork, archived, stars, android_in_topic, android_in_name_desc, android_in_readme]

    except Exception as e:
        print(f"⚠️ Error for {full_name}: {e}")
        return [None]*7

# === RUN ANALYSIS ===
for i, full_name in enumerate(df['full_name']):
    print(f"🔍 [{i+1}/{len(df)}] Checking {full_name}...")
    row = fetch_repo_data(full_name)
    results.append(row)
    sleep(0.5)  # polite delay

# === SAVE RESULTS ===
cols = ["language", "is_fork", "is_archived", "stars", "android_in_topic", "android_in_name_desc", "android_in_readme"]
df[cols] = pd.DataFrame(results, columns=cols)
df.to_csv(output_csv, index=False)
print(f"✅ Done! Metadata saved to:\n{output_csv}")


🔍 [1/25812] Checking 0015/ThatProject...
🔍 [2/25812] Checking 008chen/InterpolatorShow...
🔍 [3/25812] Checking 00ec454/Ask...
🔍 [4/25812] Checking 00ec454/pop...
🔍 [5/25812] Checking 00-Evan/shattered-pixel-dungeon...
🔍 [6/25812] Checking 06peng/FrescoDemo...
🔍 [7/25812] Checking 08carmelo/android-keeplive...
🔍 [8/25812] Checking 0maru/twitter_login...
🔍 [9/25812] Checking 0niel/university-app...
🔍 [10/25812] Checking 0ranko0P/AutoDark...
🔍 [11/25812] Checking 0x4f53/Wristkey...
🔍 [12/25812] Checking 0x5e/RubiksCubeSolver...
🔍 [13/25812] Checking 0x7c13/Pal3.Unity...
🔍 [14/25812] Checking 0xbad1d3a5/Kaku...
🔍 [15/25812] Checking 0xchat-app/0xchat-app-main...
🔍 [16/25812] Checking 0xchat-app/0xchat-core...
🔍 [17/25812] Checking 0xf104a/NextcloudServices...
🔍 [18/25812] Checking 0xkol/badspin...
🔍 [19/25812] Checking 0xm1nam0/RxCore...
🔍 [20/25812] Checking 0xZhangKe/Fread...
🔍 [21/25812] Checking 0xZhangKe/NotionLight...
🔍 [22/25812] Checking 0xZhangKe/ShiZhong...
🔍 [23/25812] Checking 